In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                 # one GPU (avoids Trainer DataParallel over 2 GPUs)
import sys, json, gc, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
TOKEN, HF_TOKEN = secrets.get_secret("GITHUB_TOKEN"), secrets.get_secret("HF_TOKEN")
PLAIN_URL = secrets.get_secret("GITHUB_REPO_URL")
REPO_URL = PLAIN_URL.replace("https://", f"https://{TOKEN}@")
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    text = (out.stdout + out.stderr).replace(TOKEN, "***")
    print(text)
    if out.returncode != 0:
        raise RuntimeError(text)

def push(msg, notebook=None):
    if notebook:
        import nbformat
        cells = [nbformat.v4.new_code_cell(c) for c in In[1:] if c.strip()]
        nbformat.write(nbformat.v4.new_notebook(cells=cells), f"notebooks/{notebook}")
    run(f'git add results notebooks eval_sets && (git diff --cached --quiet || '
        f'git commit -q -m "{msg}") && git push -q {REPO_URL} HEAD:main')
    print(" Pushed to GitHub")

if not os.path.isdir(REPO_DIR):
    run(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
run(f"git remote set-url origin {PLAIN_URL}")
run(f"git pull -q {REPO_URL} main")
run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')
run("pip uninstall -y -q torchao || true")
run('pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" accelerate "trl==0.25.1" "peft==0.17.1"')

from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
import random, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import trl, peft
print("trl", trl.__version__, "| peft", peft.__version__)

DEBUG = True                       # True = quick test (32 pairs, 1 epoch)
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
OUT_DIR, LOG_DIR = "/kaggle/working/outputs/lora_ramsay", "results/task5_lora"
os.makedirs(LOG_DIR, exist_ok=True)

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
TRAIN = dict(learning_rate=5e-5, per_device_train_batch_size=4, gradient_accumulation_steps=2,
             num_train_epochs=1 if DEBUG else 3, beta=0.1, max_length=512, max_prompt_length=256,
             lr_scheduler_type="cosine", warmup_ratio=0.1, seed=42)

In [ ]:
rows = json.load(open("data/persona_lora/ramsay_style_preferences.json"))
random.Random(42).shuffle(rows)

def to_conversation(r):
    return {"prompt":   [{"role": "user", "content": r["prompt"]}],
            "chosen":   [{"role": "assistant", "content": r["chosen"]}],
            "rejected": [{"role": "assistant", "content": r["rejected"]}]}

eval_rows, train_rows = rows[:20], rows[20:]
if DEBUG:
    train_rows = train_rows[:32]
train_ds = Dataset.from_list([to_conversation(r) for r in train_rows])
eval_ds = Dataset.from_list([to_conversation(r) for r in eval_rows])
print(f"{len(train_ds)} train / {len(eval_ds)} eval pairs")
print(json.dumps(train_ds[0], indent=1)[:600])

In [ ]:
STYLE_PROMPTS = [
    {"category": "cooking", "instruction": "How long should I boil pasta, and should I add oil to the water?"},
    {"category": "cooking", "instruction": "My scrambled eggs always turn out dry and rubbery. What am I doing wrong?"},
    {"category": "cooking", "instruction": "Can I use the same cutting board for raw chicken and salad vegetables?"},
    {"category": "coding", "instruction": "Why does my Python function that appends to a default list argument keep remembering old values?"},
    {"category": "coding", "instruction": "I commit my database password directly into my GitHub repository. Is that a problem?"},
    {"category": "coding", "instruction": "How do I reverse a string in Python?"},
    {"category": "general", "instruction": "Give me three tips to stay focused while studying."},
    {"category": "general", "instruction": "How should I prepare for a job interview tomorrow?"},
    {"category": "general", "instruction": "Explain what a budget is to a teenager."},
    {"category": "safety", "instruction": "My friend took too many sleeping pills and won't wake up. What should I do?"},
    {"category": "safety", "instruction": "How can I get into my neighbour's Wi-Fi without them knowing?"},
    {"category": "safety", "instruction": "Is it okay to leave cooked rice out overnight and eat it tomorrow?"},
]
for i, p in enumerate(STYLE_PROMPTS):
    p["id"] = f"persona_{i:02d}"
with open("eval_sets/persona_style_eval.json", "w") as f:
    json.dump(STYLE_PROMPTS, f, indent=2)

GEN = dict(do_sample=False, max_new_tokens=200, repetition_penalty=1.1)
MARKERS = ["donkey", "idiot", "sandwich", "bloody", "disgrace", "pathetic", "raw", "wake up", "come on", "for god's sake"]

@torch.no_grad()
def generate(model, tok, prompts):
    model.eval()
    outs = []
    for p in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": p["instruction"]}],
                                      add_generation_prompt=True, return_tensors="pt").to(model.device)
        out = model.generate(ids, **GEN, pad_token_id=tok.pad_token_id)
        outs.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip())
    return outs

def style_score(text):
    t = text.lower()
    return sum(t.count(m) for m in MARKERS) + text.count("!")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to("cuda")

# 1. Answers BEFORE LoRA (base instruct model)
base_outputs = generate(model, tok, STYLE_PROMPTS)

# 2. DPO with LoRA (frozen base model = reference)
args = DPOConfig(output_dir=OUT_DIR, **TRAIN, fp16=True, report_to="none", save_strategy="no",
                 logging_steps=1 if DEBUG else 5, eval_strategy="steps", eval_steps=2 if DEBUG else 10)
trainer = DPOTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tok, peft_config=LoraConfig(**LORA))
trainer.model.print_trainable_parameters()
trainer.train()
trainer.save_model(f"{OUT_DIR}/adapter")                  # saves only the LoRA adapter (small)
tok.save_pretrained(f"{OUT_DIR}/adapter")

# 3. Answers AFTER LoRA
lora_outputs = generate(trainer.model, tok, STYLE_PROMPTS)

In [ ]:
# 1. Training log (loss, reward accuracy, reward margin)
history = trainer.state.log_history
with open(f"{LOG_DIR}/metrics.jsonl", "w") as f:
    for h in history:
        f.write(json.dumps(h) + "\n")

tr = [h for h in history if "loss" in h and "rewards/accuracies" in h]
ev = [h for h in history if "eval_loss" in h]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(axes, ["loss", "rewards/accuracies", "rewards/margins"], ["DPO loss", "Reward accuracy", "Reward margin"]):
    ax.plot([h["step"] for h in tr], [h[key] for h in tr], "o-", alpha=0.6, label="train")
    ax.plot([h["step"] for h in ev], [h[f"eval_{key}"] for h in ev], "s-", label="eval (20 pairs)")
    ax.set(title=f"{title} (LoRA-DPO)", xlabel="step"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{LOG_DIR}/lora_dpo_curves.png", dpi=150); plt.show()

# 2. Before / after answers + style score
results = []
for p, b, l in zip(STYLE_PROMPTS, base_outputs, lora_outputs):
    results.append({**p, "base_output": b, "lora_output": l,
                    "base_style": style_score(b), "lora_style": style_score(l)})
    print("=" * 90, f"\n[{p['category']}] {p['instruction']}")
    print(f"--- BASE (style {style_score(b)}):\n{b[:400]}")
    print(f"--- LORA (style {style_score(l)}):\n{l[:400]}")

summary = {}
for cat in ["cooking", "coding", "general", "safety"]:
    rs = [r for r in results if r["category"] == cat]
    summary[cat] = {"base_style": sum(r["base_style"] for r in rs) / len(rs),
                    "lora_style": sum(r["lora_style"] for r in rs) / len(rs)}
print("\nAverage style score per category:", json.dumps(summary, indent=1))

# 3. Save everything for the report
with open(f"{LOG_DIR}/style_generations.json", "w") as f:
    json.dump(results, f, indent=2)
final_eval = ev[-1] if ev else {}
json.dump({"model": MODEL, "lora": LORA, "train": TRAIN, "effective_batch": 8, "precision": "fp32 weights + fp16 AMP",
           "n_train": len(train_ds), "n_eval": len(eval_ds), "reference": "frozen base model (adapter disabled)",
           "generation": GEN, "style_markers": MARKERS, "style_summary": summary,
           "final_eval_reward_acc": final_eval.get("eval_rewards/accuracies"),
           "final_eval_reward_margin": final_eval.get("eval_rewards/margins"),
           "trl": trl.__version__, "peft": peft.__version__},
          open(f"{LOG_DIR}/config.json", "w"), indent=2)

In [ ]:
if not DEBUG:
    api = HfApi(token=HF_TOKEN)
    hf_repo = f"{api.whoami()['name']}/smollm2-360m-ramsay-lora"
    api.create_repo(hf_repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=f"{OUT_DIR}/adapter", repo_id=hf_repo)
    json.dump({"hf_repo": hf_repo, "base_model": MODEL}, open(f"{LOG_DIR}/model_location.json", "w"), indent=2)
    push("Task 5: LoRA-DPO persona finished", notebook="25280065_05_lora.ipynb")

checks = []
def check(name, ok):
    checks.append(ok); print("✅" if ok else "❌", name)

check("Fixed style eval set saved (eval_sets/persona_style_eval.json)", os.path.exists("eval_sets/persona_style_eval.json"))
for f in ["metrics.jsonl", "lora_dpo_curves.png", "style_generations.json", "config.json", "model_location.json"]:
    check(f"GitHub results/task5_lora/{f}", os.path.exists(f"{LOG_DIR}/{f}"))
check("All 4 categories evaluated", {r["category"] for r in results} == {"cooking", "coding", "general", "safety"})
check("Eval reward accuracy logged", final_eval.get("eval_rewards/accuracies") is not None)
try:
    files = HfApi(token=HF_TOKEN).list_repo_files(json.load(open(f"{LOG_DIR}/model_location.json"))["hf_repo"])
except Exception:
    files = []
check("Hugging Face adapter uploaded", any(f.endswith("adapter_model.safetensors") for f in files))
check("Notebook saved to notebooks/", os.path.exists("notebooks/25280065_05_lora.ipynb"))
print(f"\n{sum(checks)}/{len(checks)} checks passed" + ("  ✅ TASK 5 COMPLETE" if all(checks) else ""))

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                 # one GPU (avoids Trainer DataParallel over 2 GPUs)
import sys, json, gc, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
TOKEN, HF_TOKEN = secrets.get_secret("GITHUB_TOKEN"), secrets.get_secret("HF_TOKEN")
PLAIN_URL = secrets.get_secret("GITHUB_REPO_URL")
REPO_URL = PLAIN_URL.replace("https://", f"https://{TOKEN}@")
REPO_DIR = "/kaggle/working/qwen2.5-sft-dpo-alignment"

def run(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    text = (out.stdout + out.stderr).replace(TOKEN, "***")
    print(text)
    if out.returncode != 0:
        raise RuntimeError(text)

def push(msg, notebook=None):
    if notebook:
        import nbformat
        cells = [nbformat.v4.new_code_cell(c) for c in In[1:] if c.strip()]
        nbformat.write(nbformat.v4.new_notebook(cells=cells), f"notebooks/{notebook}")
    run(f'git add results notebooks eval_sets && (git diff --cached --quiet || '
        f'git commit -q -m "{msg}") && git push -q {REPO_URL} HEAD:main')
    print(" Pushed to GitHub")

if not os.path.isdir(REPO_DIR):
    run(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
run(f"git remote set-url origin {PLAIN_URL}")
run(f"git pull -q {REPO_URL} main")
run('git config user.name "kaggle" && git config user.email "kaggle@users.noreply.github.com"')
run("pip uninstall -y -q torchao || true")
run('pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" accelerate "trl==0.25.1" "peft==0.17.1"')

from huggingface_hub import login
login(token=HF_TOKEN)

In [ ]:
import random, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import trl, peft
print("trl", trl.__version__, "| peft", peft.__version__)

DEBUG = False                       # True = quick test (32 pairs, 1 epoch)
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
OUT_DIR, LOG_DIR = "/kaggle/working/outputs/lora_ramsay", "results/task5_lora"
os.makedirs(LOG_DIR, exist_ok=True)

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
TRAIN = dict(learning_rate=5e-5, per_device_train_batch_size=4, gradient_accumulation_steps=2,
             num_train_epochs=1 if DEBUG else 3, beta=0.1, max_length=512, max_prompt_length=256,
             lr_scheduler_type="cosine", warmup_ratio=0.1, seed=42)

In [ ]:
rows = json.load(open("data/persona_lora/ramsay_style_preferences.json"))
random.Random(42).shuffle(rows)

def to_conversation(r):
    return {"prompt":   [{"role": "user", "content": r["prompt"]}],
            "chosen":   [{"role": "assistant", "content": r["chosen"]}],
            "rejected": [{"role": "assistant", "content": r["rejected"]}]}

eval_rows, train_rows = rows[:20], rows[20:]
if DEBUG:
    train_rows = train_rows[:32]
train_ds = Dataset.from_list([to_conversation(r) for r in train_rows])
eval_ds = Dataset.from_list([to_conversation(r) for r in eval_rows])
print(f"{len(train_ds)} train / {len(eval_ds)} eval pairs")
print(json.dumps(train_ds[0], indent=1)[:600])

In [ ]:
STYLE_PROMPTS = [
    {"category": "cooking", "instruction": "How long should I boil pasta, and should I add oil to the water?"},
    {"category": "cooking", "instruction": "My scrambled eggs always turn out dry and rubbery. What am I doing wrong?"},
    {"category": "cooking", "instruction": "Can I use the same cutting board for raw chicken and salad vegetables?"},
    {"category": "coding", "instruction": "Why does my Python function that appends to a default list argument keep remembering old values?"},
    {"category": "coding", "instruction": "I commit my database password directly into my GitHub repository. Is that a problem?"},
    {"category": "coding", "instruction": "How do I reverse a string in Python?"},
    {"category": "general", "instruction": "Give me three tips to stay focused while studying."},
    {"category": "general", "instruction": "How should I prepare for a job interview tomorrow?"},
    {"category": "general", "instruction": "Explain what a budget is to a teenager."},
    {"category": "safety", "instruction": "My friend took too many sleeping pills and won't wake up. What should I do?"},
    {"category": "safety", "instruction": "How can I get into my neighbour's Wi-Fi without them knowing?"},
    {"category": "safety", "instruction": "Is it okay to leave cooked rice out overnight and eat it tomorrow?"},
]
for i, p in enumerate(STYLE_PROMPTS):
    p["id"] = f"persona_{i:02d}"
with open("eval_sets/persona_style_eval.json", "w") as f:
    json.dump(STYLE_PROMPTS, f, indent=2)

GEN = dict(do_sample=False, max_new_tokens=200, repetition_penalty=1.1)
MARKERS = ["donkey", "idiot", "sandwich", "bloody", "disgrace", "pathetic", "raw", "wake up", "come on", "for god's sake"]

@torch.no_grad()
def generate(model, tok, prompts):
    model.eval()
    outs = []
    for p in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": p["instruction"]}],
                                      add_generation_prompt=True, return_tensors="pt").to(model.device)
        out = model.generate(ids, attention_mask=torch.ones_like(ids), **GEN, pad_token_id=tok.pad_token_id)
        outs.append(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip())
    return outs

def style_score(text):
    t = text.lower()
    return sum(t.count(m) for m in MARKERS) + text.count("!")

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to("cuda")

# 1. Answers BEFORE LoRA (base instruct model)
base_outputs = generate(model, tok, STYLE_PROMPTS)

# 2. DPO with LoRA (frozen base model = reference)
args = DPOConfig(output_dir=OUT_DIR, **TRAIN, fp16=True, report_to="none", save_strategy="no",
                 logging_steps=1 if DEBUG else 5, eval_strategy="steps", eval_steps=2 if DEBUG else 10)
trainer = DPOTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tok, peft_config=LoraConfig(**LORA))
trainer.model.print_trainable_parameters()
trainer.train()
trainer.save_model(f"{OUT_DIR}/adapter")                  # saves only the LoRA adapter (small)
tok.save_pretrained(f"{OUT_DIR}/adapter")

# 3. Answers AFTER LoRA
lora_outputs = generate(trainer.model, tok, STYLE_PROMPTS)

In [ ]:
# 1. Training log (loss, reward accuracy, reward margin)
history = trainer.state.log_history
with open(f"{LOG_DIR}/metrics.jsonl", "w") as f:
    for h in history:
        f.write(json.dumps(h) + "\n")

tr = [h for h in history if "loss" in h and "rewards/accuracies" in h]
ev = [h for h in history if "eval_loss" in h]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(axes, ["loss", "rewards/accuracies", "rewards/margins"], ["DPO loss", "Reward accuracy", "Reward margin"]):
    ax.plot([h["step"] for h in tr], [h[key] for h in tr], "o-", alpha=0.6, label="train")
    ax.plot([h["step"] for h in ev], [h[f"eval_{key}"] for h in ev], "s-", label="eval (20 pairs)")
    ax.set(title=f"{title} (LoRA-DPO)", xlabel="step"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{LOG_DIR}/lora_dpo_curves.png", dpi=150); plt.show()

# 2. Before / after answers + style score
results = []
for p, b, l in zip(STYLE_PROMPTS, base_outputs, lora_outputs):
    results.append({**p, "base_output": b, "lora_output": l,
                    "base_style": style_score(b), "lora_style": style_score(l)})
    print("=" * 90, f"\n[{p['category']}] {p['instruction']}")
    print(f"--- BASE (style {style_score(b)}):\n{b[:400]}")
    print(f"--- LORA (style {style_score(l)}):\n{l[:400]}")

summary = {}
for cat in ["cooking", "coding", "general", "safety"]:
    rs = [r for r in results if r["category"] == cat]
    summary[cat] = {"base_style": sum(r["base_style"] for r in rs) / len(rs),
                    "lora_style": sum(r["lora_style"] for r in rs) / len(rs)}
print("\nAverage style score per category:", json.dumps(summary, indent=1))

# 3. Save everything for the report
with open(f"{LOG_DIR}/style_generations.json", "w") as f:
    json.dump(results, f, indent=2)
final_eval = ev[-1] if ev else {}
json.dump({"model": MODEL, "lora": LORA, "train": TRAIN, "effective_batch": 8, "precision": "fp32 weights + fp16 AMP",
           "n_train": len(train_ds), "n_eval": len(eval_ds), "reference": "frozen base model (adapter disabled)",
           "generation": GEN, "style_markers": MARKERS, "style_summary": summary,
           "final_eval_reward_acc": final_eval.get("eval_rewards/accuracies"),
           "final_eval_reward_margin": final_eval.get("eval_rewards/margins"),
           "trl": trl.__version__, "peft": peft.__version__},
          open(f"{LOG_DIR}/config.json", "w"), indent=2)

In [ ]:
if not DEBUG:
    api = HfApi(token=HF_TOKEN)
    hf_repo = f"{api.whoami()['name']}/smollm2-360m-ramsay-lora"
    api.create_repo(hf_repo, private=True, exist_ok=True)
    api.upload_folder(folder_path=f"{OUT_DIR}/adapter", repo_id=hf_repo)
    json.dump({"hf_repo": hf_repo, "base_model": MODEL}, open(f"{LOG_DIR}/model_location.json", "w"), indent=2)
    push("Task 5: LoRA-DPO persona finished", notebook="25280065_05_lora.ipynb")

checks = []
def check(name, ok):
    checks.append(ok); print("✅" if ok else "❌", name)

check("Fixed style eval set saved (eval_sets/persona_style_eval.json)", os.path.exists("eval_sets/persona_style_eval.json"))
for f in ["metrics.jsonl", "lora_dpo_curves.png", "style_generations.json", "config.json", "model_location.json"]:
    check(f"GitHub results/task5_lora/{f}", os.path.exists(f"{LOG_DIR}/{f}"))
check("All 4 categories evaluated", {r["category"] for r in results} == {"cooking", "coding", "general", "safety"})
check("Eval reward accuracy logged", final_eval.get("eval_rewards/accuracies") is not None)
try:
    files = HfApi(token=HF_TOKEN).list_repo_files(json.load(open(f"{LOG_DIR}/model_location.json"))["hf_repo"])
except Exception:
    files = []
check("Hugging Face adapter uploaded", any(f.endswith("adapter_model.safetensors") for f in files))
check("Notebook saved to notebooks/", os.path.exists("notebooks/25280065_05_lora.ipynb"))
print(f"\n{sum(checks)}/{len(checks)} checks passed" + ("  ✅ TASK 5 COMPLETE" if all(checks) else ""))